In [48]:
from reasonable_crowd.dataset import load_annotations, build_evaluation_dataset
import os
import numpy as np
path_to_reasonable_crowd = "../../Reasonable-Crowd"
path_to_workers = os.path.join(path_to_reasonable_crowd, "annotations/workers.txt")

with open(path_to_workers, "r") as f:
    workers = f.readlines()


annotations = load_annotations(path_to_reasonable_crowd)
X, y, y_votes, y_agreement = build_evaluation_dataset(annotations)

In [49]:
num_pairs = 0
for scenario, pairs in annotations.items():
    num_pairs += len(pairs)
print("Number of pairs:", num_pairs)

Number of pairs: 3364


In [76]:
worker_preferences = {}
for worker in workers:
    w_id = worker.strip()
    worker_preferences[w_id] = set()
    
for scenario, pairs in annotations.items():
    for pair_id, votes in pairs.items():
        for w_id, preferences in worker_preferences.items():
            if w_id in votes:
                t1, t2 = pair_id.split(" ;; " )
                preferences.add((t1, t2))
                

In [ ]:
agreements = []
for w_id, preferences in worker_preferences.items():
    pairs = set()
    for pref in preferences:
        pairs.add(pref)
        pairs.add(pref[::-1])
        
    for other_w_id, other_preferences in worker_preferences.items():
        if w_id == other_w_id:
            continue
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
        total_count = len(pairs.intersection(other_pairs))//2
        common_preferences = preferences.intersection(other_preferences)
        agreement_count = len(common_preferences)
        agreement = agreement_count / total_count if total_count > 0 else 0
        agreements.append(agreement)
agreements.sort()

agreements = np.array(agreements)

print("Annotator agreement statistics:")
print("Min agreement:", np.min(agreements))
print("Max agreement:", np.max(agreements))
print("Mean agreement:", np.mean(agreements))
print("Median agreement:", np.median(agreements))
print(len(agreements), "annotator pairs compared.")

Annotator agreement statistics:
Min agreement: 0.0
Max agreement: 1.0
Mean agreement: 0.5826184387946431
Median agreement: 0.7777777777777778
4160 annotator pairs compared.


In [88]:
from rulebook_benchmark.rulebook import Relation

accuracies = []

for w_id, preferences in worker_preferences.items():
    correct = 0
    total = 0
    for pair, label in zip(X, y):
        t1, t2 = pair
        if (t1, t2) in preferences:
            decision = Relation.LARGER
        elif (t2, t1) in preferences:
            decision = Relation.SMALLER
        else:
            continue
    
        total += 1
        if decision == label:
            correct += 1
    accuracy = correct / total if total > 0 else 0
    accuracies.append(accuracy)
    
accuracies = np.array(accuracies)
accuracies.sort()

print("Annotator accuracy statistics:")
print("Min accuracy:", np.min(accuracies))
print("Max accuracy:", np.max(accuracies))
print("Mean accuracy:", np.mean(accuracies))
print("Median accuracy:", np.median(accuracies))
print(len(accuracies), "annotators evaluated.")
            

Annotator accuracy statistics:
Min accuracy: 0.5
Max accuracy: 1.0
Mean accuracy: 0.8363332215110086
Median accuracy: 0.8429319371727748
65 annotators evaluated.
